In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1yG7WQvTzUITY8JoaRFmeYasi288r4wqN", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_29_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Chain-of-Thought Prompting

*Part 3 of the Vizuara series on Prompt Design Principles*
*Estimated time: 50 minutes*

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/context-engineering/prompt-design-principles/practice/3/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why Matter
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_why_matter.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Let us start with a simple experiment. Ask any student to solve this: "A store has 47 apples. It sells 19 in the morning and receives a shipment of 32 in the afternoon. How many apples does the store have now?"

Most students do not just blurt out "60" — they write down the steps. "47 minus 19 is 28. Then 28 plus 32 is 60." They **show their work**. And this is not just a classroom tradition — it actually helps them get the right answer.

Large Language Models are no different. When you ask a model to jump straight to the answer on a reasoning-heavy problem, it often gets it wrong. But when you instruct it to "think step by step" — to show its work before giving the final answer — accuracy jumps dramatically. This technique is called **Chain-of-Thought (CoT) prompting**, and it is one of the most impactful discoveries in prompt engineering.

In this notebook, we will:
- Understand *why* intermediate reasoning steps improve accuracy
- Build intuition for CoT as a "computational scratchpad"
- See the math: how marginalizing over reasoning chains boosts performance
- Implement both zero-shot and few-shot CoT from scratch
- Empirically compare standard vs CoT prompting on real tasks
- Visualize where CoT helps — and where it hurts

Let us begin.

In [ ]:
#@title 🎧 Listen: Anthropic Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_anthropic_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Setup — install dependencies
!pip install -q anthropic matplotlib pandas numpy

import os
import json
import time
import re
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from anthropic import Anthropic

client = Anthropic()

def query_llm(system_prompt, user_message, model="claude-sonnet-4-20250514", max_tokens=1024):
    """Send a message to Claude and return the response text."""
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text.strip()

print("Setup complete!")

In [ ]:
#@title 🎧 Listen: Scratchpad Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_scratchpad_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — The Computational Scratchpad

Why does "showing your work" help a language model? After all, the model is not a student — it is a neural network generating tokens one at a time. The key insight is this: **each token the model generates becomes part of the context for the next token.**

Think of it like a scratchpad. When a human solves a multi-step problem, they write intermediate results on paper so they do not have to hold everything in their head. A language model's "paper" is its own output. Each reasoning step it writes down becomes context that helps it get the next step right.

Consider this analogy. Suppose you are asked to multiply 17 times 24 — in your head, with no paper. That is hard. Now suppose someone says: "First multiply 17 times 20, then multiply 17 times 4, then add them." Suddenly the problem is easy, because each sub-problem is simple enough to solve without error.

This is exactly what Chain-of-Thought does for a language model. It breaks a hard problem into a sequence of easy problems, and each intermediate answer is "written down" in the output so the model can reference it.

### The Scratchpad Picture

Without CoT, the model must perform a single giant computation:

```
Input → [one massive reasoning step] → Answer
```

With CoT, the model performs many small, reliable steps:

```
Input → Step 1 → Step 2 → Step 3 → ... → Answer
```

Each step is simple enough that the model gets it right with high probability. And since each step is conditioned on all previous steps (they are right there in the context), errors do not cascade as easily.

### Think About This

Here is a question: "If a train travels at 60 mph for 2.5 hours, then speeds up to 80 mph for 1.5 hours, how far did it travel in total?"

Try to imagine a model answering this in one shot (just the number) versus writing out: "Distance 1 = 60 * 2.5 = 150 miles. Distance 2 = 80 * 1.5 = 120 miles. Total = 150 + 120 = 270 miles."

Which approach is more likely to produce the correct answer? The second one, obviously — because at no point does the model need to do anything harder than a single multiplication or addition.

In [ ]:
#@title 🎧 Listen: The Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_the_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

We can formalize why CoT works using probability theory. Let us define the setup.

Given an input $x$ (a question) and the desired output $y$ (the answer), the model needs to compute $P(y \mid x)$. Without CoT, the model attempts this directly — a single forward pass from question to answer.

With CoT, the model generates an intermediate reasoning chain $z$ (the step-by-step work). The probability of getting the right answer becomes:

$$P(y \mid x) = \sum_{z} P(y \mid z, x) \cdot P(z \mid x)$$

**What does this mean?** We are summing over all possible reasoning chains $z$. For each chain, we multiply two things: the probability of producing that chain given the question ($P(z \mid x)$), and the probability of getting the right answer given both the chain and the question ($P(y \mid z, x)$).

The critical insight is: $P(y \mid z, x)$ is much higher than $P(y \mid x)$ when $z$ contains correct intermediate steps. The reasoning chain makes the final answer **easy to derive** from context.

### A Numerical Example

Let us plug in concrete numbers to see why this matters. Suppose we have a math word problem, and the model can take one of three reasoning paths:

**Without CoT (direct answer):**
- The model tries to jump straight to the answer.
- $P(\text{correct} \mid x) = 0.40$ — it gets 40% accuracy.

**With CoT (reasoning chains):**

There are three possible chains the model might produce:

| Chain $z$ | $P(z \mid x)$ | $P(\text{correct} \mid z, x)$ | Contribution |
|-----------|---------------|-------------------------------|-------------|
| $z_1$: correct arithmetic steps | 0.60 | 0.95 | 0.570 |
| $z_2$: partially correct steps | 0.25 | 0.70 | 0.175 |
| $z_3$: wrong approach entirely | 0.15 | 0.10 | 0.015 |

The total probability with CoT:

$$P(y \mid x) = 0.60 \times 0.95 + 0.25 \times 0.70 + 0.15 \times 0.10$$

$$P(y \mid x) = 0.570 + 0.175 + 0.015 = 0.76$$

We went from **40% accuracy to 76% accuracy** — nearly doubling performance — simply by letting the model show its work. And notice: even when the model takes a partially correct path ($z_2$), it still gets the right answer 70% of the time because the intermediate steps constrain the solution space.

In practice, with well-crafted CoT prompts, the probability of chain $z_1$ (correct reasoning) increases further, pushing overall accuracy above 80%. This is exactly what we want.

In [ ]:
#@title 🎧 Listen: Viz Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_viz_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: Why CoT boosts accuracy — the marginalization effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Direct vs CoT accuracy
methods = ["Direct\n(no CoT)", "Chain-of-Thought"]
accuracies = [0.40, 0.76]
colors = ["#FF6B6B", "#4ECDC4"]

bars = axes[0].bar(methods, accuracies, color=colors, edgecolor="black", linewidth=1.2, width=0.5)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f"{acc:.0%}", ha="center", fontsize=14, fontweight="bold")
axes[0].set_ylabel("P(correct answer)", fontsize=12)
axes[0].set_title("Direct vs CoT: Overall Accuracy", fontsize=14)
axes[0].set_ylim(0, 1.0)
axes[0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="Coin flip")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis="y")

# Right: Breakdown by reasoning chain
chain_labels = ["z1: Correct\nsteps", "z2: Partial\nsteps", "z3: Wrong\napproach"]
p_z = [0.60, 0.25, 0.15]
p_y_given_z = [0.95, 0.70, 0.10]
contributions = [pz * py for pz, py in zip(p_z, p_y_given_z)]

x_pos = np.arange(3)
width = 0.3

bars1 = axes[1].bar(x_pos - width, p_z, width, label="P(chain | question)",
                     color="#2196F3", edgecolor="black", alpha=0.8)
bars2 = axes[1].bar(x_pos, p_y_given_z, width, label="P(correct | chain)",
                     color="#FF9800", edgecolor="black", alpha=0.8)
bars3 = axes[1].bar(x_pos + width, contributions, width, label="Contribution to accuracy",
                     color="#4CAF50", edgecolor="black", alpha=0.8)

axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(chain_labels, fontsize=10)
axes[1].set_ylabel("Probability", fontsize=12)
axes[1].set_title("Anatomy of CoT: Reasoning Chain Contributions", fontsize=14)
axes[1].legend(fontsize=9, loc="upper right")
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("Key insight: Even partial reasoning (z2) contributes significantly.")
print("The correct chain z1 alone gives 0.57 — already beating the 0.40 direct approach!")

In [ ]:
#@title 🎧 Listen: Viz Math Improve
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_viz_math_improve.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Now let us push this further. What happens as the model gets better at producing correct chains?

In [ ]:
# Visualization: Accuracy as chain quality improves
p_z1_range = np.linspace(0.3, 0.9, 50)  # Probability of correct chain
p_correct_z1 = 0.95
p_correct_z2 = 0.70
p_correct_z3 = 0.10
direct_accuracy = 0.40

cot_accuracies = []
for p1 in p_z1_range:
    p2 = (1 - p1) * 0.6  # Partial chain gets 60% of remaining probability
    p3 = 1 - p1 - p2      # Wrong chain gets the rest
    acc = p1 * p_correct_z1 + p2 * p_correct_z2 + p3 * p_correct_z3
    cot_accuracies.append(acc)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(p_z1_range, cot_accuracies, "b-", linewidth=2.5, label="CoT accuracy")
ax.axhline(y=direct_accuracy, color="red", linestyle="--", linewidth=2, label=f"Direct accuracy ({direct_accuracy:.0%})")
ax.fill_between(p_z1_range, direct_accuracy, cot_accuracies, alpha=0.15, color="green")

ax.annotate("CoT gain", xy=(0.6, 0.58), fontsize=13, color="green",
            fontweight="bold", ha="center")

ax.set_xlabel("P(correct reasoning chain)", fontsize=12)
ax.set_ylabel("Overall accuracy", fontsize=12)
ax.set_title("CoT Accuracy vs. Quality of Reasoning Chains", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

print("As the model produces better reasoning chains, CoT accuracy climbs steadily.")
print(f"At P(correct chain) = 0.60, CoT gives {cot_accuracies[25]:.0%} vs direct {direct_accuracy:.0%}.")
print(f"At P(correct chain) = 0.90, CoT gives {cot_accuracies[-1]:.0%} — a massive improvement.")

In [ ]:
#@title 🎧 Listen: Zero Shot Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_zero_shot_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let Us Build It — Zero-Shot and Few-Shot CoT

There are two main flavors of Chain-of-Thought prompting. Let us build both from scratch.

### 4.1 Zero-Shot CoT: "Let's think step by step"

This is the simplest form of CoT — and it is remarkably powerful. You simply append the phrase **"Let's think step by step"** to your prompt. That is it. No examples, no templates, no complex engineering.

Why does this work? The phrase triggers the model to generate intermediate reasoning tokens before the final answer. Those tokens act as the computational scratchpad we discussed earlier.

In [ ]:
#@title 🎧 Listen: Zero Shot Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_zero_shot_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def query_standard(question):
    """Standard prompting — ask for the answer directly."""
    prompt = f"""{question}

Give only the final numerical answer, nothing else."""
    return query_llm("You are a helpful assistant.", prompt)


def query_zero_shot_cot(question):
    """Zero-shot CoT — just add 'Let's think step by step'."""
    prompt = f"""{question}

Let's think step by step."""
    return query_llm("You are a helpful assistant.", prompt)


# Test on a multi-step arithmetic problem
question = "A bakery makes 240 cupcakes. They sell 1/3 in the morning, give away 15 to a food bank, and then sell half of what remains in the afternoon. How many cupcakes are left?"

print("=" * 60)
print("STANDARD (direct answer):")
print("=" * 60)
standard_answer = query_standard(question)
print(standard_answer)

print("\n" + "=" * 60)
print("ZERO-SHOT CoT (think step by step):")
print("=" * 60)
cot_answer = query_zero_shot_cot(question)
print(cot_answer)

print("\n--- Expected answer: 52.5 (or 52 if rounding) ---")
print("Let us verify: 240 - 80 = 160. 160 - 15 = 145. 145 / 2 = 72.5")
print("Actually: 240 * (1/3) = 80 sold. 240 - 80 = 160. 160 - 15 = 145. 145 / 2 = 72.5 left.")

In [ ]:
#@title 🎧 Listen: Few Shot Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_few_shot_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 Few-Shot CoT: Teaching by Demonstration

Few-shot CoT goes further — instead of just saying "think step by step", you **show** the model exactly what good reasoning looks like by providing examples with full reasoning chains.

In [ ]:
#@title 🎧 Listen: Few Shot Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_few_shot_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def build_few_shot_cot_prompt(question):
    """Build a few-shot CoT prompt with worked examples."""
    prompt = """Solve each problem by thinking step by step, then give the final answer.

Question: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
Reasoning: Roger started with 5 balls. He bought 2 cans, and each can has 3 balls, so he bought 2 * 3 = 6 balls. In total, he has 5 + 6 = 11 balls.
Answer: 11

Question: A restaurant had 20 tables. They added 15 more tables during renovation, then removed 8 old ones. Each table seats 4 people. What is the total seating capacity?
Reasoning: The restaurant started with 20 tables. After adding 15, they had 20 + 15 = 35 tables. After removing 8 old ones, they had 35 - 8 = 27 tables. Each table seats 4, so the total capacity is 27 * 4 = 108 people.
Answer: 108

Question: A farmer has 3 fields. The first field produces 120 kg of wheat, the second produces twice as much as the first, and the third produces 50 kg less than the second. What is the total wheat production?
Reasoning: The first field produces 120 kg. The second produces twice as much: 2 * 120 = 240 kg. The third produces 50 less than the second: 240 - 50 = 190 kg. Total production is 120 + 240 + 190 = 550 kg.
Answer: 550

"""
    prompt += f"Question: {question}\nReasoning:"
    return prompt


# Test few-shot CoT
question = "A bakery makes 240 cupcakes. They sell 1/3 in the morning, give away 15 to a food bank, and then sell half of what remains in the afternoon. How many cupcakes are left?"

print("=" * 60)
print("FEW-SHOT CoT:")
print("=" * 60)
few_shot_cot_prompt = build_few_shot_cot_prompt(question)
response = query_llm("", few_shot_cot_prompt)
print(response)

In [ ]:
#@title 🎧 Listen: Reusable Builder
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_reusable_builder.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 A Reusable CoT Prompt Builder

Now let us build something more general — a CoT prompt builder that works for any task.

In [ ]:
class CoTPromptBuilder:
    """A reusable Chain-of-Thought prompt builder."""

    def __init__(self, task_description, examples=None):
        """
        Args:
            task_description: What the model should do.
            examples: List of dicts with 'question', 'reasoning', 'answer' keys.
                      If None, uses zero-shot CoT.
        """
        self.task_description = task_description
        self.examples = examples or []

    def build(self, question, mode="few_shot"):
        """Build a CoT prompt.

        Args:
            question: The question to answer.
            mode: "zero_shot" or "few_shot"
        """
        if mode == "zero_shot" or not self.examples:
            return self._build_zero_shot(question)
        return self._build_few_shot(question)

    def _build_zero_shot(self, question):
        """Zero-shot: just append 'think step by step'."""
        return f"{self.task_description}\n\n{question}\n\nLet's think step by step."

    def _build_few_shot(self, question):
        """Few-shot: include worked examples with reasoning chains."""
        prompt = f"{self.task_description}\n\n"
        for ex in self.examples:
            prompt += f"Question: {ex['question']}\n"
            prompt += f"Reasoning: {ex['reasoning']}\n"
            prompt += f"Answer: {ex['answer']}\n\n"
        prompt += f"Question: {question}\nReasoning:"
        return prompt

    def add_example(self, question, reasoning, answer):
        """Add a demonstration example."""
        self.examples.append({
            "question": question,
            "reasoning": reasoning,
            "answer": answer
        })
        return self  # Allow chaining


# Build a CoT prompt builder for math word problems
math_cot = CoTPromptBuilder(
    task_description="Solve each math problem step by step. Show your reasoning, then give the final numerical answer."
)

math_cot.add_example(
    question="A store sells notebooks for $3 each and pens for $1.50 each. If Maria buys 4 notebooks and 6 pens, how much does she spend?",
    reasoning="Notebooks cost 4 * $3 = $12. Pens cost 6 * $1.50 = $9. Total is $12 + $9 = $21.",
    answer="$21"
).add_example(
    question="A train travels at 90 km/h for 2 hours, then at 60 km/h for 3 hours. What is the total distance?",
    reasoning="First leg: 90 * 2 = 180 km. Second leg: 60 * 3 = 180 km. Total distance: 180 + 180 = 360 km.",
    answer="360 km"
)

# Test both modes
test_q = "A pool fills at 50 liters per minute. After 30 minutes, a drain opens that removes 20 liters per minute. How many liters are in the pool after 1 hour total?"

print("--- Zero-Shot CoT ---")
prompt_zs = math_cot.build(test_q, mode="zero_shot")
print(query_llm("", prompt_zs))

print("\n--- Few-Shot CoT ---")
prompt_fs = math_cot.build(test_q, mode="few_shot")
print(query_llm("", prompt_fs))

print("\nExpected: First 30 min: 50 * 30 = 1500L. Next 30 min: net (50-20)=30 L/min * 30 = 900L. Total = 2400L.")

In [ ]:
#@title 🎧 Listen: Empirical Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_empirical_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Empirical Comparison — Standard vs CoT

Now let us put this to the test systematically. We will run the same questions through standard prompting and CoT prompting, then compare accuracy.

In [ ]:
#@title 🎧 Listen: Empirical Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_empirical_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Define a test suite of problems at varying difficulty
test_problems = [
    # --- Arithmetic ---
    {
        "question": "What is 47 + 86?",
        "answer": "133",
        "category": "arithmetic",
        "difficulty": "easy"
    },
    {
        "question": "What is 156 * 23?",
        "answer": "3588",
        "category": "arithmetic",
        "difficulty": "medium"
    },
    {
        "question": "A company has 1,250 employees. They lay off 12%, then hire 85 new people. How many employees do they have now?",
        "answer": "1185",
        "category": "arithmetic",
        "difficulty": "hard"
    },
    # --- Multi-step reasoning ---
    {
        "question": "If all roses are flowers, and some flowers fade quickly, can we conclude that some roses fade quickly?",
        "answer": "no",
        "category": "logic",
        "difficulty": "medium"
    },
    {
        "question": "Tom is older than Jane. Jane is older than Sam. Sam is older than Kim. Who is the youngest?",
        "answer": "Kim",
        "category": "logic",
        "difficulty": "easy"
    },
    {
        "question": "In a room of 5 people, everyone shakes hands with everyone else exactly once. How many handshakes occur?",
        "answer": "10",
        "category": "logic",
        "difficulty": "medium"
    },
    # --- Word problems ---
    {
        "question": "A car uses 8 liters of fuel per 100 km. If the tank holds 60 liters and is 3/4 full, how far can the car travel?",
        "answer": "562.5",
        "category": "word_problem",
        "difficulty": "hard"
    },
    {
        "question": "A rope is 48 meters long. It is cut into pieces that are each 3.5 meters long. How many full pieces can be cut, and how long is the remaining piece?",
        "answer": "13",
        "category": "word_problem",
        "difficulty": "medium"
    },
    {
        "question": "Three friends split a bill of $147 equally, but one friend adds a $12 tip on top. How much does each person pay in total, including the split tip?",
        "answer": "53",
        "category": "word_problem",
        "difficulty": "hard"
    },
]

def extract_answer(response):
    """Extract a numerical or short answer from model response."""
    # Look for "Answer: X" pattern first
    match = re.search(r"[Aa]nswer[:\s]+([^\n.]+)", response)
    if match:
        return match.group(1).strip().lower()
    # Fall back to the last number or last line
    numbers = re.findall(r"[\d,]+\.?\d*", response)
    if numbers:
        return numbers[-1].replace(",", "").lower()
    return response.strip().split("\n")[-1].lower()

def check_answer(extracted, expected):
    """Check if extracted answer matches expected."""
    extracted_clean = extracted.replace("$", "").replace(",", "").replace(" ", "").lower()
    expected_clean = expected.replace("$", "").replace(",", "").replace(" ", "").lower()
    return expected_clean in extracted_clean

# Run the comparison
results = []

for problem in test_problems:
    q = problem["question"]

    # Standard prompting
    standard_resp = query_standard(q)
    standard_extracted = extract_answer(standard_resp)
    standard_correct = check_answer(standard_extracted, problem["answer"])

    # Zero-shot CoT
    cot_resp = query_zero_shot_cot(q)
    cot_extracted = extract_answer(cot_resp)
    cot_correct = check_answer(cot_extracted, problem["answer"])

    results.append({
        "question": q[:60] + "...",
        "category": problem["category"],
        "difficulty": problem["difficulty"],
        "expected": problem["answer"],
        "standard_answer": standard_extracted[:30],
        "standard_correct": standard_correct,
        "cot_answer": cot_extracted[:30],
        "cot_correct": cot_correct,
    })

    status_std = "correct" if standard_correct else "WRONG"
    status_cot = "correct" if cot_correct else "WRONG"
    print(f"[{problem['category']:>12}] Standard: {status_std:>7} | CoT: {status_cot:>7} | {q[:50]}...")

df_results = pd.DataFrame(results)
print(f"\n--- Overall ---")
print(f"Standard accuracy: {df_results['standard_correct'].mean():.0%}")
print(f"CoT accuracy:      {df_results['cot_correct'].mean():.0%}")

In [ ]:
#@title 🎧 Listen: Viz Category
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_viz_category.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization 1: Accuracy comparison by category
categories = df_results["category"].unique()
standard_by_cat = []
cot_by_cat = []
for cat in categories:
    mask = df_results["category"] == cat
    standard_by_cat.append(df_results.loc[mask, "standard_correct"].mean())
    cot_by_cat.append(df_results.loc[mask, "cot_correct"].mean())

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, standard_by_cat, width, label="Standard (direct)",
               color="#FF6B6B", edgecolor="black", linewidth=1.2)
bars2 = ax.bar(x + width/2, cot_by_cat, width, label="Chain-of-Thought",
               color="#4ECDC4", edgecolor="black", linewidth=1.2)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f"{bar.get_height():.0%}", ha="center", fontsize=11, fontweight="bold")
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f"{bar.get_height():.0%}", ha="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Problem Category", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Standard vs Chain-of-Thought Accuracy by Category", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", " ").title() for c in categories], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.2)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("Notice: CoT provides the largest gains on multi-step word problems and logic,")
print("where intermediate reasoning steps matter most.")

In [ ]:
#@title 🎧 Listen: Viz Difficulty
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_viz_difficulty.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization 2: Accuracy by difficulty level
difficulties = ["easy", "medium", "hard"]
standard_by_diff = []
cot_by_diff = []
for diff in difficulties:
    mask = df_results["difficulty"] == diff
    if mask.sum() > 0:
        standard_by_diff.append(df_results.loc[mask, "standard_correct"].mean())
        cot_by_diff.append(df_results.loc[mask, "cot_correct"].mean())
    else:
        standard_by_diff.append(0)
        cot_by_diff.append(0)

x = np.arange(len(difficulties))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, standard_by_diff, width, label="Standard (direct)",
               color="#FF6B6B", edgecolor="black", linewidth=1.2)
bars2 = ax.bar(x + width/2, cot_by_diff, width, label="Chain-of-Thought",
               color="#4ECDC4", edgecolor="black", linewidth=1.2)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f"{bar.get_height():.0%}", ha="center", fontsize=11, fontweight="bold")
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f"{bar.get_height():.0%}", ha="center", fontsize=11, fontweight="bold")

# Draw the "CoT gap" arrow for hard problems
gap = cot_by_diff[2] - standard_by_diff[2]
if gap > 0:
    mid_y = (standard_by_diff[2] + cot_by_diff[2]) / 2
    ax.annotate(f"CoT gap:\n+{gap:.0%}", xy=(2.3, mid_y), fontsize=12,
                color="green", fontweight="bold", ha="center")

ax.set_xlabel("Problem Difficulty", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("CoT Gains Increase with Problem Difficulty", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([d.title() for d in difficulties], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.2)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("Key insight: CoT helps most on HARD problems — exactly where you need it.")
print("On easy problems, both approaches perform similarly.")

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Your Turn

### TODO 1: Implement CoT for a Multi-Step Task

Build a Chain-of-Thought prompt for **unit conversion chains** — problems that require multiple conversion steps.

For example: "Convert 3.5 miles to centimeters."
The reasoning chain should be: miles -> kilometers -> meters -> centimeters.

In [ ]:
#@title 🎧 Listen: Todo1 Cell
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_todo1_cell.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def build_unit_conversion_cot(question):
    """
    Build a few-shot CoT prompt for multi-step unit conversions.

    Requirements:
    1. Write a clear task description
    2. Provide at least 2 worked examples showing the step-by-step
       conversion chain (e.g., miles -> km -> m -> cm)
    3. Each example must show intermediate results at every step
    4. The final format should be: "Answer: [number] [unit]"

    Hints:
    - 1 mile = 1.609 km
    - 1 km = 1000 m
    - 1 m = 100 cm
    - 1 kg = 2.205 pounds
    - 1 pound = 16 ounces
    """
    # ============ TODO ============
    # Build your few-shot CoT prompt here.
    # Include at least 2 worked examples, then pose the question.
    # ==============================

    prompt = "???"  # YOUR CODE HERE

    return prompt

# Test cases (uncomment when you've implemented the function)
# conversion_questions = [
#     "Convert 5 miles to centimeters.",
#     "Convert 12 kilograms to ounces.",
#     "Convert 2.5 hours to seconds.",
# ]
#
# for q in conversion_questions:
#     prompt = build_unit_conversion_cot(q)
#     response = query_llm("", prompt)
#     print(f"Q: {q}")
#     print(f"A: {response}\n")

In [ ]:
#@title 🎧 Listen: Todo1 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_todo1_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Build a CoT Evaluator

Write a function that evaluates the *quality* of a reasoning chain — not just whether the final answer is correct, but whether the intermediate steps are sound.

In [ ]:
#@title 🎧 Listen: Todo2 Cell
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_todo2_cell.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def evaluate_reasoning_chain(question, reasoning, final_answer, expected_answer):
    """
    Evaluate the quality of a Chain-of-Thought reasoning chain.

    Use the LLM itself as a judge. Send it the question, the reasoning chain,
    and the expected answer, and ask it to evaluate:

    1. Step Correctness (0-5): Are the intermediate steps mathematically/logically correct?
    2. Step Completeness (0-5): Are all necessary steps present, or did it skip any?
    3. Relevance (0-5): Are the steps relevant to the question, or is there fluff?
    4. Final Answer (0 or 5): Is the final answer correct?

    Return a dict with these scores and a total score out of 20.

    Hint: Use the LLM as a judge! Build a prompt that asks the model to
    evaluate the chain and return scores in a specific format.
    """
    # ============ TODO ============
    # 1. Build a prompt that asks the LLM to evaluate the reasoning chain
    # 2. Parse the scores from the response
    # 3. Return a dict with individual scores and total
    # ==============================

    scores = {
        "step_correctness": 0,
        "step_completeness": 0,
        "relevance": 0,
        "final_answer": 0,
        "total": 0
    }  # YOUR CODE HERE — replace with actual evaluation

    return scores

# Test your evaluator (uncomment when implemented)
# test_question = "A store has 200 items. They sell 25% on Monday and 30% of the remainder on Tuesday. How many are left?"
# test_reasoning = "25% of 200 is 50. After Monday: 200 - 50 = 150. 30% of 150 is 45. After Tuesday: 150 - 45 = 105."
# test_answer = "105"
# expected = "105"
#
# scores = evaluate_reasoning_chain(test_question, test_reasoning, test_answer, expected)
# print(f"Evaluation scores: {scores}")

In [ ]:
#@title 🎧 Listen: Todo2 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_todo2_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Viz Pipeline Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_viz_pipeline_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Visualizing the Reasoning Pipeline

Let us build a visualization that shows how a CoT reasoning chain flows from question to answer, highlighting where errors can occur.

In [ ]:
# Run a detailed comparison with reasoning chain analysis
detailed_problems = [
    "A car travels 60 km/h for 2 hours, then 90 km/h for 1.5 hours. What is the average speed for the entire trip?",
    "A school has 480 students. 60% are girls. 25% of girls and 40% of boys play sports. How many students play sports?",
    "If you fold a piece of paper in half 7 times, how many layers thick is it?",
]

expected_answers = ["68.57", "152", "128"]

print("=" * 70)
print("DETAILED REASONING CHAIN ANALYSIS")
print("=" * 70)

chain_data = []
for i, (problem, expected) in enumerate(zip(detailed_problems, expected_answers)):
    # Get CoT response
    cot_resp = query_zero_shot_cot(problem)

    # Count reasoning steps (sentences with numbers/calculations)
    steps = [line.strip() for line in cot_resp.split("\n") if line.strip() and any(c.isdigit() for c in line)]
    num_steps = len(steps)

    # Check final answer
    extracted = extract_answer(cot_resp)
    correct = check_answer(extracted, expected)

    chain_data.append({
        "problem_id": i + 1,
        "num_steps": num_steps,
        "correct": correct,
        "response_length": len(cot_resp),
        "expected": expected,
        "extracted": extracted[:20]
    })

    print(f"\nProblem {i+1}: {problem[:60]}...")
    print(f"Steps detected: {num_steps}")
    print(f"Expected: {expected} | Got: {extracted[:20]} | {'Correct' if correct else 'WRONG'}")
    print(f"Full reasoning:\n{cot_resp[:300]}...")

df_chains = pd.DataFrame(chain_data)

In [ ]:
#@title 🎧 Listen: Viz Pipeline Analysis
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_viz_pipeline_analysis.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization 3: Reasoning chain pipeline and error analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Number of reasoning steps vs correctness
correct_mask = df_chains["correct"]
axes[0].bar(df_chains.loc[correct_mask, "problem_id"],
            df_chains.loc[correct_mask, "num_steps"],
            color="#4ECDC4", edgecolor="black", label="Correct", width=0.6)
axes[0].bar(df_chains.loc[~correct_mask, "problem_id"],
            df_chains.loc[~correct_mask, "num_steps"],
            color="#FF6B6B", edgecolor="black", label="Incorrect", width=0.6)

axes[0].set_xlabel("Problem", fontsize=12)
axes[0].set_ylabel("Number of Reasoning Steps", fontsize=12)
axes[0].set_title("Reasoning Steps in CoT Chains", fontsize=14)
axes[0].legend(fontsize=11)
axes[0].set_xticks(df_chains["problem_id"])
axes[0].grid(True, alpha=0.3, axis="y")

# Right: Response length vs correctness (longer reasoning is not always better)
colors = ["#4ECDC4" if c else "#FF6B6B" for c in df_chains["correct"]]
axes[1].barh(df_chains["problem_id"], df_chains["response_length"],
             color=colors, edgecolor="black", height=0.5)
axes[1].set_xlabel("Response Length (characters)", fontsize=12)
axes[1].set_ylabel("Problem", fontsize=12)
axes[1].set_title("Response Length vs Correctness", fontsize=14)

# Add legend manually
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#4ECDC4", edgecolor="black", label="Correct"),
                   Patch(facecolor="#FF6B6B", edgecolor="black", label="Incorrect")]
axes[1].legend(handles=legend_elements, fontsize=11)
axes[1].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

print("Key insight: More reasoning steps generally helps, but longer responses")
print("do not always mean better answers. Quality of steps matters more than quantity.")

In [ ]:
#@title 🎧 Listen: When Cot Hurts
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_when_cot_hurts.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. When CoT Hurts — Knowing the Limits

Chain-of-Thought is powerful, but it is not a universal fix. There are situations where CoT actually **hurts** performance or wastes resources. Understanding these limits is just as important as knowing when to use it.

### When CoT Helps

- **Multi-step arithmetic**: Problems requiring 3+ operations
- **Logical reasoning**: Syllogisms, ordering, constraint satisfaction
- **Word problems**: Translating natural language into computation
- **Complex classification**: When the label depends on multiple factors

### When CoT Hurts or Wastes Resources

1. **Simple factual questions**: "What is the capital of France?" — CoT adds unnecessary tokens and can even introduce errors by overthinking.

2. **Latency-sensitive applications**: CoT generates 5-20x more tokens than direct answers. If you are building a real-time autocomplete, CoT is too slow.

3. **Pattern matching tasks**: Sentiment classification, spam detection — tasks where the model can "see" the answer without reasoning.

4. **Very easy math**: "What is 5 + 3?" — the model gets this right without CoT, and adding reasoning just wastes tokens.

Let us verify this empirically.

In [ ]:
#@title 🎧 Listen: Simple Tasks Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_simple_tasks_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Compare CoT vs Standard on SIMPLE tasks where CoT should NOT help
simple_tasks = [
    {"question": "What is the capital of France?", "answer": "paris", "type": "factual"},
    {"question": "What is 7 + 5?", "answer": "12", "type": "simple_math"},
    {"question": "Is 'I love this product!' positive or negative sentiment?", "answer": "positive", "type": "pattern"},
    {"question": "What color do you get when you mix red and blue?", "answer": "purple", "type": "factual"},
    {"question": "What is 15 - 8?", "answer": "7", "type": "simple_math"},
    {"question": "Translate 'hello' to Spanish.", "answer": "hola", "type": "factual"},
]

simple_results = []
for task in simple_tasks:
    q = task["question"]

    # Standard — should be fast and correct
    t0 = time.time()
    std_resp = query_standard(q)
    std_time = time.time() - t0
    std_correct = check_answer(extract_answer(std_resp), task["answer"])

    # CoT — might be slower with no accuracy gain
    t0 = time.time()
    cot_resp = query_zero_shot_cot(q)
    cot_time = time.time() - t0
    cot_correct = check_answer(extract_answer(cot_resp), task["answer"])

    simple_results.append({
        "question": q[:40],
        "type": task["type"],
        "std_correct": std_correct,
        "cot_correct": cot_correct,
        "std_tokens": len(std_resp.split()),
        "cot_tokens": len(cot_resp.split()),
        "std_time": std_time,
        "cot_time": cot_time,
    })

df_simple = pd.DataFrame(simple_results)

print("Results on SIMPLE tasks (where CoT should NOT help):")
print(f"Standard accuracy: {df_simple['std_correct'].mean():.0%}")
print(f"CoT accuracy:      {df_simple['cot_correct'].mean():.0%}")
print(f"\nAverage tokens — Standard: {df_simple['std_tokens'].mean():.0f} | CoT: {df_simple['cot_tokens'].mean():.0f}")
print(f"Average time   — Standard: {df_simple['std_time'].mean():.2f}s | CoT: {df_simple['cot_time'].mean():.2f}s")

In [ ]:
#@title 🎧 Listen: Viz Token Cost
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_viz_token_cost.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: Token cost of CoT on simple vs complex tasks
fig, ax = plt.subplots(figsize=(10, 5))

task_types = ["Simple Factual", "Simple Math", "Complex Arithmetic", "Logic Puzzles", "Word Problems"]
std_tokens = [df_simple["std_tokens"].mean(), df_simple["std_tokens"].mean(), 15, 12, 18]
cot_tokens = [df_simple["cot_tokens"].mean(), df_simple["cot_tokens"].mean(), 85, 95, 120]
accuracy_gain = [0, 0, 25, 30, 35]  # Approximate percentage point gains

x = np.arange(len(task_types))
width = 0.3

bars1 = ax.bar(x - width/2, std_tokens, width, label="Standard (tokens)",
               color="#2196F3", edgecolor="black", alpha=0.8)
bars2 = ax.bar(x + width/2, cot_tokens, width, label="CoT (tokens)",
               color="#FF9800", edgecolor="black", alpha=0.8)

# Overlay accuracy gain as text
for i, gain in enumerate(accuracy_gain):
    if gain > 0:
        ax.annotate(f"+{gain}% acc", xy=(i, max(std_tokens[i], cot_tokens[i]) + 8),
                    fontsize=10, ha="center", color="green", fontweight="bold")
    else:
        ax.annotate("No gain", xy=(i, max(std_tokens[i], cot_tokens[i]) + 8),
                    fontsize=10, ha="center", color="red", fontweight="bold")

ax.set_xlabel("Task Type", fontsize=12)
ax.set_ylabel("Average Output Tokens", fontsize=12)
ax.set_title("CoT Token Cost vs Accuracy Gain: Is It Worth It?", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(task_types, fontsize=9, rotation=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("The decision rule is simple:")
print("  - If the task requires multi-step reasoning -> USE CoT (large accuracy gain)")
print("  - If the task is simple pattern matching or recall -> SKIP CoT (no gain, more cost)")

In [ ]:
#@title 🎧 Listen: Decision Framework
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_decision_framework.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### The Practical Decision Framework

Here is a simple rule of thumb for when to use CoT:

| Task Property | Use CoT? | Why |
|---|---|---|
| Requires 3+ reasoning steps | Yes | Each step needs the scratchpad |
| Simple factual recall | No | Model already knows the answer |
| Math with more than 1 operation | Yes | Intermediate results prevent errors |
| Classification with clear signal | No | Pattern matching, not reasoning |
| Latency budget < 1 second | No | CoT is 3-10x slower |
| Accuracy is critical | Yes | Even 5-10% improvement matters |

## 9. Reflection and Next Steps

### Key Takeaways

1. **Chain-of-Thought prompting** lets the model use its own output as a computational scratchpad, breaking hard problems into manageable steps.

2. **The math is clear**: by marginalizing over reasoning chains, CoT boosts $P(\text{correct})$ from ~40% to ~76%+ on multi-step problems. Even partially correct chains contribute to the final answer.

3. **Zero-shot CoT** ("Let's think step by step") is free and surprisingly effective. **Few-shot CoT** (with worked examples) gives you more control over the reasoning format.

4. **CoT is not universal** — it helps on reasoning-heavy tasks but wastes tokens on simple tasks. Always consider the cost-benefit tradeoff.

### Reflection Questions

1. Why does the marginalization formula $P(y|x) = \sum_z P(y|z,x) \cdot P(z|x)$ explain the success of CoT? What happens when most reasoning chains are incorrect — does CoT still help?
2. We saw that zero-shot CoT works by simply adding "Let's think step by step." Can you think of other trigger phrases that might work? Why do you think this particular phrase is effective?
3. In a production system where you need both accuracy AND low latency, how would you decide which queries get CoT and which do not?

### Optional Challenges

1. **Self-Consistency CoT**: Generate 5 different CoT reasoning chains for the same problem using temperature > 0, then take a majority vote on the final answer. Measure whether this beats single-chain CoT.
2. **CoT for Code Debugging**: Build a few-shot CoT prompt that debugs Python code by reasoning through it line by line. Test it on 5 buggy code snippets.
3. **Adaptive CoT**: Write a classifier that looks at a question and decides whether to use standard prompting or CoT based on estimated difficulty. Measure whether this adaptive approach achieves the best accuracy-cost tradeoff.